# BoundaryCells — locked paired utility
20 arms, fixed20schedule, train-only group holdouts; no official test. Five arms resume audited calibrationr2. Controls fresh. No target/HP/gate changes. Session budget can stage incomplete arms; no utility PASS before all controls/r20. Mandatory online W&B/private inline key persists in private version history.

In [1]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.1 which is incompatible.
wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] 

W&B authenticated online


In [2]:
MH=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'model_packet.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'teacher_base.py', 'teacher_driver.py', 'v10_energy.py', 'v10_run.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py', 'v5_anchor.py', 'v5_books.py', 'v5_run.py', 'v6_geometry.py', 'v6_run.py', 'v6_schema.py', 'v7_basis.py', 'v7_nystrom.py', 'v7_run.py', 'v8_hybrid.py', 'v8_run.py', 'v9_anchor.py', 'v9_run.py']
PROJ=['boundary_driver.py', 'boundary_method.py', 'cells.py', 'probe.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
assert torch.cuda.device_count()==2
assert all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('boundary_driver.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0];WORK=Path('/kaggle/working/work');OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source');SOURCE.mkdir(exist_ok=True)
for folder in ('mh','knowledge','papers/pilot/fixture','papers/boundary-cells/proj'):
    (WORK/folder).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
for path in SRC.iterdir():
    if path.name in MH: dst=WORK/'mh'/path.name
    elif path.name in PROJ: dst=WORK/'papers/boundary-cells/proj'/path.name
    elif path.name=='meta.json': dst=WORK/'knowledge'/path.name
    elif path.name=='contract.md': continue
    else: dst=WORK/'papers/pilot/fixture'/path.name
    shutil.copyfile(path,dst)
EXPECTED={'bc_single_s43': {'fingerprint': 'a31f484f81d61ae1a50add87bdc41ea1f287d58c54a043bd20b2e162d3a5f352', 'source': {'mh/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mh/archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'mh/channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'mh/dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'mh/metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'mh/mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'mh/model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'mh/new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'mh/new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'mh/p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'mh/pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'mh/prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'mh/teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'mh/teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'mh/v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'mh/v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'mh/v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'mh/v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'mh/v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'mh/v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'mh/v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'mh/v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'mh/v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'mh/v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'mh/v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'mh/v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'mh/v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'mh/v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'mh/v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'mh/v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'mh/v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'mh/v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'mh/v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', 'mh/v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'mh/v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'mh/v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'mh/v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mh/v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'mh/v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'mh/v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'proj/boundary_driver.py': 'bebe94c6c2483235d271fbb75d794df81d59533b4a88a9478193a14ffb96c19d', 'proj/boundary_method.py': '456945b64182dcdb2dfa41d5a42a6cea37f36667606d018ae3b2df47398c9530', 'proj/cells.py': '456e0400470a0f5f76b3808b71d4e256535d3e62f6f0e1ceddf2278925875fe0', 'proj/probe.py': 'a50a7d62e2c105ab12107e76de797acd7ab9ce5c7924451782866f4dc8a304e6'}, 'fixture': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'last_round': 2}, 'bc_direct2_fl_s43': {'fingerprint': '2fcd446207971392f82a22002bbee3c03bb80f0ffb5981df10cbb16ef36146cf', 'source': {'mh/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mh/archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'mh/channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'mh/dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'mh/metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'mh/mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'mh/model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'mh/new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'mh/new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'mh/p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'mh/pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'mh/prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'mh/teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'mh/teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'mh/v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'mh/v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'mh/v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'mh/v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'mh/v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'mh/v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'mh/v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'mh/v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'mh/v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'mh/v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'mh/v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'mh/v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'mh/v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'mh/v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'mh/v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'mh/v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'mh/v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'mh/v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'mh/v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', 'mh/v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'mh/v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'mh/v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'mh/v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mh/v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'mh/v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'mh/v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'proj/boundary_driver.py': 'bebe94c6c2483235d271fbb75d794df81d59533b4a88a9478193a14ffb96c19d', 'proj/boundary_method.py': '456945b64182dcdb2dfa41d5a42a6cea37f36667606d018ae3b2df47398c9530', 'proj/cells.py': '456e0400470a0f5f76b3808b71d4e256535d3e62f6f0e1ceddf2278925875fe0', 'proj/probe.py': 'a50a7d62e2c105ab12107e76de797acd7ab9ce5c7924451782866f4dc8a304e6'}, 'fixture': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'last_round': 2}, 'bc_direct4_fl_s43': {'fingerprint': '1e33c74329df39eb1c0be1851a565acde38447ce25f33c0051f5abfb72e2d470', 'source': {'mh/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mh/archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'mh/channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'mh/dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'mh/metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'mh/mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'mh/model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'mh/new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'mh/new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'mh/p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'mh/pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'mh/prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'mh/teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'mh/teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'mh/v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'mh/v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'mh/v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'mh/v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'mh/v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'mh/v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'mh/v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'mh/v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'mh/v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'mh/v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'mh/v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'mh/v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'mh/v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'mh/v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'mh/v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'mh/v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'mh/v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'mh/v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'mh/v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', 'mh/v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'mh/v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'mh/v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'mh/v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mh/v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'mh/v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'mh/v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'proj/boundary_driver.py': 'bebe94c6c2483235d271fbb75d794df81d59533b4a88a9478193a14ffb96c19d', 'proj/boundary_method.py': '456945b64182dcdb2dfa41d5a42a6cea37f36667606d018ae3b2df47398c9530', 'proj/cells.py': '456e0400470a0f5f76b3808b71d4e256535d3e62f6f0e1ceddf2278925875fe0', 'proj/probe.py': 'a50a7d62e2c105ab12107e76de797acd7ab9ce5c7924451782866f4dc8a304e6'}, 'fixture': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'last_round': 2}, 'bc_teacher2_fl_s43': {'fingerprint': '7b09d2e325ac07f555deb8ca791350957f3208e50eaab0e43bd172154d4db3d8', 'source': {'mh/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mh/archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'mh/channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'mh/dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'mh/metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'mh/mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'mh/model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'mh/new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'mh/new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'mh/p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'mh/pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'mh/prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'mh/teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'mh/teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'mh/v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'mh/v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'mh/v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'mh/v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'mh/v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'mh/v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'mh/v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'mh/v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'mh/v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'mh/v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'mh/v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'mh/v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'mh/v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'mh/v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'mh/v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'mh/v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'mh/v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'mh/v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'mh/v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', 'mh/v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'mh/v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'mh/v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'mh/v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mh/v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'mh/v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'mh/v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'proj/boundary_driver.py': 'bebe94c6c2483235d271fbb75d794df81d59533b4a88a9478193a14ffb96c19d', 'proj/boundary_method.py': '456945b64182dcdb2dfa41d5a42a6cea37f36667606d018ae3b2df47398c9530', 'proj/cells.py': '456e0400470a0f5f76b3808b71d4e256535d3e62f6f0e1ceddf2278925875fe0', 'proj/probe.py': 'a50a7d62e2c105ab12107e76de797acd7ab9ce5c7924451782866f4dc8a304e6'}, 'fixture': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'last_round': 2}, 'bc_teacher4_fl_s43': {'fingerprint': '14a7ae2b7fb5e512222e619cb9f0c4bb7a024f77dda90db5724b48ab4d0bcf95', 'source': {'mh/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mh/archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'mh/channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'mh/dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'mh/metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'mh/mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'mh/model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'mh/new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'mh/new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'mh/p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'mh/pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'mh/prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'mh/teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'mh/teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'mh/v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'mh/v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'mh/v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'mh/v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'mh/v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'mh/v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'mh/v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'mh/v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'mh/v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'mh/v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'mh/v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'mh/v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'mh/v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'mh/v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'mh/v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'mh/v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'mh/v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'mh/v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'mh/v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', 'mh/v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'mh/v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'mh/v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'mh/v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mh/v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'mh/v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'mh/v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'proj/boundary_driver.py': 'bebe94c6c2483235d271fbb75d794df81d59533b4a88a9478193a14ffb96c19d', 'proj/boundary_method.py': '456945b64182dcdb2dfa41d5a42a6cea37f36667606d018ae3b2df47398c9530', 'proj/cells.py': '456e0400470a0f5f76b3808b71d4e256535d3e62f6f0e1ceddf2278925875fe0', 'proj/probe.py': 'a50a7d62e2c105ab12107e76de797acd7ab9ce5c7924451782866f4dc8a304e6'}, 'fixture': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'last_round': 2}}
ADAPTER='"""Prevent preflight from advancing restored W&B steps; model source stays frozen."""\nimport runpy\nimport sys\nfrom pathlib import Path\n\n\ndef install(wandb):\n    run_class = wandb.sdk.wandb_run.Run\n    original = run_class.log\n\n    def log(self, data, *args, **kwargs):\n        if set(data) == {\'preflight/online\'} and not args and \'step\' not in kwargs:\n            kwargs = dict(kwargs, commit=False)\n        return original(self, data, *args, **kwargs)\n\n    run_class.log = log\n\n\nif __name__ == \'__main__\':\n    import wandb\n    install(wandb)\n    driver = sys.argv.pop(1)\n    sys.argv[0] = driver\n    sys.path.insert(0, str(Path(driver).parent))\n    runpy.run_path(driver, run_name=\'__main__\')\n'
IMPORTER='"""Import the unique audited BoundaryCells parent, publish complete markers last."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef import_run(mount, output, name, expected, manifest, verify):\n    candidates = [p.parent for p in Path(mount).rglob(\'config.json\')\n                  if p.parent.name == name and\n                  json.loads(p.read_text()).get(\'fingerprint\') == expected[\'fingerprint\']]\n    assert len(candidates) == 1, (\'require-resume missing/ambiguous\', name, candidates)\n    source = candidates[0]\n    identity = json.loads((source / \'config.json\').read_text())\n    assert identity[\'source\'] == expected[\'source\'] and identity[\'fixture\'] == expected[\'fixture\']\n    audit = verify(source, manifest, expected[\'fixture\'], expected[\'last_round\'], calibration=True)\n    assert audit[\'cost\'][\'pass_cost\'], \'cost FAIL cannot open utility\'\n    destination = Path(output) / name\n    files = [p for p in source.rglob(\'*\') if p.is_file() and\n             \'wandb\' not in p.relative_to(source).parts and p.name != \'log.txt\']\n    for marker_pass in (False, True):\n        for path in files:\n            relative = path.relative_to(source)\n            if (relative.parts[0] == \'complete\') != marker_pass:\n                continue\n            target = destination / relative\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if target.exists():\n                assert sha(target) == sha(path), (\'resume collision\', relative)\n                continue\n            temporary = target.with_name(target.name + \'.import-part\')\n            shutil.copyfile(path, temporary)\n            assert sha(temporary) == sha(path)\n            os.replace(temporary, target)\n    assert verify(destination, manifest, expected[\'fixture\'], expected[\'last_round\'], calibration=True)[\'rounds\'] == expected[\'last_round\']\n    return dict(run=name, last_round=expected[\'last_round\'], fingerprint=expected[\'fingerprint\'],\n                copied_files=len(files), markers_last=True)\n'
VERIFIER='"""Terminal pull and audit for BoundaryCells calibration; never launch or stop training."""\nimport ast\nimport fcntl\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\nimport nbformat\nimport numpy as np\nimport torch\n\nROOT=Path(__file__).resolve().parents[2]\nBASE=ROOT/\'papers/boundary-cells\'\nHELPER=ROOT/\'.agents/skills/kaggle-training-notebook/scripts\'\nsys.path.insert(0,str(ROOT))\nfrom mh.metrics import METRIC_KEYS,metrics_from_confusion,per_class_from_confusion\nfrom mh.dagsnet import build_model,CFG\n\n\ndef sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef verify(run, expected, fixture_sha, expected_rounds=2, calibration=True):\n    identity=json.loads((run/\'config.json\').read_text())\n    fp=identity.pop(\'fingerprint\')\n    assert hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()==fp\n    assert identity[\'fixture\']==fixture_sha and identity[\'config\'][\'rounds\']==20\n    assert identity[\'config\'][\'seed\'] in (42,43) and not identity[\'config\'][\'smoke\']\n    for name,digest in identity[\'source\'].items():assert expected[name.split(\'/\',1)[1]]==digest\n    rows=sorted((run/\'complete\').glob(\'round_*.json\'))\n    assert len(rows)==expected_rounds, (\'wrong committed round count\',len(rows),expected_rounds)\n    classes=json.loads((ROOT/\'knowledge/meta.json\').read_text())[\'class_names\']\n    previous=None\n    metric_checks=0\n    for r,file in enumerate(rows,1):\n        row=json.loads(file.read_text());assert row[\'round\']==r and row[\'fingerprint\']==fp and row[\'prev_sha\']==previous\n        for folder,key,ext in [(\'weights\',\'weights_sha\',\'pt\'),(\'resume\',\'resume_sha\',\'pt\'),(\'predictions\',\'predictions_sha\',\'npz\')]:\n            assert sha(run/folder/f\'round_{r:03d}.{ext}\')==row[key]\n        if \'payload_sha\' in row:\n            assert sha(run/\'knowledge\'/f\'round_{r:03d}.bin\')==row[\'payload_sha\']\n        previous=row[\'weights_sha\']\n        weights=torch.load(run/\'weights\'/f\'round_{r:03d}.pt\',map_location=\'cpu\',weights_only=True)\n        resume=torch.load(run/\'resume\'/f\'round_{r:03d}.pt\',map_location=\'cpu\',weights_only=True)\n        assert weights[\'round\']==resume[\'round\']==r\n        assert len(weights[\'private\'])==len(row[\'clients\'])==20\n        for sd in weights[\'private\']+weights[\'proxy\']:\n            assert all(torch.isfinite(v).all() for v in sd.values() if v.is_floating_point())\n        mass=resume[\'algorithm\'][\'mass\'];logs=resume[\'algorithm\'][\'log_pi_effective\']\n        for k,item in enumerate(row[\'clients\']):\n            assert item[\'applied\']>0 and item[\'skipped\']<=16\n            prior=mass[k]/item[\'applied\']+1./item[\'rows\']\n            assert torch.allclose((prior/prior.sum()).log().float(),logs[k],atol=1e-6,rtol=1e-6)\n        with np.load(run/\'predictions\'/f\'round_{r:03d}.npz\') as pred:\n            for role,holds in row[\'eval\'].items():\n                for hold,rules in holds.items():\n                    y=pred[hold+\'_y\'];assert np.isin(y,np.arange(16)).all()\n                    for rule,ev in rules.items():\n                        values=pred[f\'{role}_{hold}_{rule}\'];assert values.shape==(20,len(y))\n                        cms=np.stack([np.bincount(y.astype(\'int64\')*16+p,minlength=256).reshape(16,16) for p in values])\n                        assert np.array_equal(cms,np.asarray(ev[\'confusion_clients\']))\n                        mm=[metrics_from_confusion(c) for c in cms]\n                        for key in METRIC_KEYS:\n                            assert np.isclose(np.mean([m[key] for m in mm]),ev[\'mean_metrics\'][key],atol=1e-12,rtol=1e-12)\n                            metric_checks+=1\n                        pc=np.mean([[p[\'f1\'] for p in per_class_from_confusion(c,classes)] for c in cms],axis=0)\n                        assert np.allclose(pc,ev[\'f1_per_class\'],atol=1e-12,rtol=1e-12)\n                        assert np.isclose(pc[0],ev[\'benign_f1\'],atol=1e-12,rtol=1e-12)\n        model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).eval()\n        model.load_state_dict(weights[\'private\'][0],strict=True)\n        assert sum(p.numel() for p in model.parameters())==395024\n        del model,weights,resume\n    cost=None\n    if calibration:\n        cost=json.loads((run/\'cost_gate.json\').read_text())\n        assert np.allclose(cost[\'algorithm_seconds\'],[json.loads(f.read_text())[\'algorithm_seconds\'] for f in rows])\n        assert np.isclose(cost[\'mean_seconds\'],np.mean(cost[\'algorithm_seconds\']))\n        assert np.isclose(cost[\'ratio\'],cost[\'mean_seconds\']/cost[\'baseline_mean_seconds\'])\n        assert cost[\'pass_cost\']==(cost[\'ratio\']<=2)\n    return dict(run=run.name,rounds=len(rows),metric_checks=metric_checks,cost=cost,\n                checkpoint_strict_rebuild=True,sampled_saved_prediction_inference=False,utility_r20=False)\n\n\ndef collect(folder):\n    metadata=json.loads((folder/\'kernel-metadata.json\').read_text());kernel=metadata[\'id\'];owner=kernel.split(\'/\')[0]\n    dest=BASE/\'kaggle\'/folder.name;marker=dest/\'AUDITED.json\'\n    if marker.exists():return True\n    prefix=[sys.executable,str(HELPER/\'kaggle_as.py\'),owner,\'--\']\n    result=subprocess.run(prefix+[\'kaggle\',\'kernels\',\'status\',kernel],capture_output=True,text=True,check=True)\n    print(result.stdout.strip(),flush=True)\n    if not any(word in result.stdout for word in (\'COMPLETE\',\'ERROR\')):return False\n    subprocess.run(prefix+[sys.executable,str(HELPER/\'pull_kernel_output.py\'),kernel,\'-p\',str(dest),\'-j\',\'2\',\n                   \'--pattern\',r\'^(runs|source)/\'],check=True)\n    private=BASE/\'private_push\'/folder.name\n    executed=private/\'executed.ipynb\'\n    subprocess.run(prefix+[sys.executable,str(HELPER/\'download_executed_notebook.py\'),kernel,\'--version\',\'1\',\n                   \'--output\',str(executed)],check=True,umask=0o077)\n    executed.chmod(0o600)\n    reference=nbformat.read(folder/metadata[\'code_file\'],as_version=4)\n    pushed=nbformat.read(private/metadata[\'code_file\'],as_version=4)\n    key=next(n.value.value for n in ast.parse(pushed.cells[1].source).body if isinstance(n,ast.Assign) and\n             any(isinstance(t,ast.Name) and t.id==\'_INLINE_WANDB_KEY\' for t in n.targets))\n    actual=nbformat.reads(executed.read_text().replace(key,\'[REDACTED]\'),as_version=4)\n    assert len(actual.cells)==len(reference.cells)\n    for i,cell in enumerate(actual.cells):\n        if cell.cell_type==\'code\':assert cell.source==pushed.cells[i].source.replace(key,\'[REDACTED]\')\n    actual.cells[1].source=reference.cells[1].source\n    actual.metadata[\'credential_cell_redacted\']=True\n    nbformat.write(actual,folder/\'executed.redacted.ipynb\')\n    expected=json.loads((folder/\'source_manifest.json\').read_text())\n    source=dest/\'source\'\n    assert json.loads((source/\'source_manifest.json\').read_text())==expected\n    for name,digest in expected.items():\n        assert sha(source/name)==digest\n        assert key not in (source/name).read_text(errors=\'ignore\')\n    fixture_sha=sha(source/\'fixture_manifest.json\')\n    path=dest/\'runs/calibration_summary.json\'\n    assert path.exists(), \'terminal output missing calibration summary; inspect failure before retry\'\n    summary=json.loads(path.read_text());assert len(summary)==5\n    audits=[]\n    with (ROOT/\'papers/.artifact-verification.lock\').open(\'a\') as lock:\n        fcntl.flock(lock,fcntl.LOCK_EX)\n        for name,item in summary.items():\n            assert item[\'calibration_valid\'] and item[\'rc\']==0\n            audits.append(verify(dest/\'runs\'/name,expected,fixture_sha))\n    for seed in {int(name.rsplit(\'_s\',1)[1]) for name in summary}:\n        baseline=next(a[\'cost\'][\'mean_seconds\'] for a in audits if a[\'run\']==f\'bc_single_s{seed}\')\n        assert all(np.isclose(a[\'cost\'][\'baseline_mean_seconds\'],baseline) for a in audits if a[\'run\'].endswith(f\'_s{seed}\'))\n    record=dict(kernel=kernel,version=1,source=expected,summary=summary,audits=audits,\n                calibration_valid=True,utility_r20=False)\n    tmp=marker.with_suffix(\'.tmp\');tmp.write_text(json.dumps(record,indent=2)+\'\\n\');tmp.replace(marker)\n    print(\'AUDITED CALIBRATION\',kernel,\'utility r20 unmeasured\',flush=True)\n    return True\n\n\nif __name__==\'__main__\':\n    torch.set_num_threads(1)\n    folders=sorted(p.parent for p in (BASE/\'notebook\').glob(\'bc-cal-s4?-20261013-initial2-v1/PUSHED\'))\n    assert len(folders)==2\n    while True:\n        results=[collect(folder) for folder in folders]\n        if all(results):break\n        time.sleep(55)\n'

(WORK/'boundary_logging.py').write_text(ADAPTER)
(WORK/'boundary_resume.py').write_text(IMPORTER)
(WORK/'scripts/kaggle').mkdir(parents=True,exist_ok=True)
(WORK/'scripts/kaggle/collect_boundary_calibration.py').write_text(VERIFIER)
for name,body in [('boundary_logging.py',ADAPTER),('boundary_resume.py',IMPORTER),('boundary_verify.py',VERIFIER)]:
    (SOURCE/name).write_text(body)
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'papers/boundary-cells/proj'))
from boundary_resume import import_run
from scripts.kaggle.collect_boundary_calibration import verify
imports=[import_run('/kaggle/input',OUT,name,value,manifest,verify) for name,value in EXPECTED.items()]
(SOURCE/'resume-import.json').write_text(json.dumps(imports,indent=2))
print('REQUIRE RESUME / COST PASS before fixture decoding',imports,flush=True)

fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('fixture mismatch',name)
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
print('source/fixture verified; CUDA',torch.__version__,flush=True)
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'papers/boundary-cells/proj'))
from mh.dagsnet import build_model,CFG
from cells import packet,decode
import numpy as np
for gpu in (0,1):
    torch.cuda.set_device(gpu);torch.manual_seed(42)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(f'cuda:{gpu}').eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy',mmap_mode='r')[::100][:32].astype('float32')).to(f'cuda:{gpu}')
    z=model(x)[0];assert torch.isfinite(z).all()
    z.square().mean().backward();assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    identity=bytes(16);encoded=packet(1,2,b'preflight',identity);assert decode(encoded,1,2,identity)==b'preflight'
    print('T4 forward/backward/codec PASS',gpu,flush=True)
    del model,x,z;torch.cuda.empty_cache()


REQUIRE RESUME / COST PASS before fixture decoding [{'run': 'bc_single_s43', 'last_round': 2, 'fingerprint': 'a31f484f81d61ae1a50add87bdc41ea1f287d58c54a043bd20b2e162d3a5f352', 'copied_files': 12, 'markers_last': True}, {'run': 'bc_direct2_fl_s43', 'last_round': 2, 'fingerprint': '2fcd446207971392f82a22002bbee3c03bb80f0ffb5981df10cbb16ef36146cf', 'copied_files': 14, 'markers_last': True}, {'run': 'bc_direct4_fl_s43', 'last_round': 2, 'fingerprint': '1e33c74329df39eb1c0be1851a565acde38447ce25f33c0051f5abfb72e2d470', 'copied_files': 14, 'markers_last': True}, {'run': 'bc_teacher2_fl_s43', 'last_round': 2, 'fingerprint': '7b09d2e325ac07f555deb8ca791350957f3208e50eaab0e43bd172154d4db3d8', 'copied_files': 14, 'markers_last': True}, {'run': 'bc_teacher4_fl_s43', 'last_round': 2, 'fingerprint': '14a7ae2b7fb5e512222e619cb9f0c4bb7a024f77dda90db5724b48ab4d0bcf95', 'copied_files': 14, 'markers_last': True}]
source/fixture verified; CUDA 2.10.0+cu128
T4 forward/backward/codec PASS 0
T4 forward/bac

In [3]:
SEED=43
SESSION_HOURS=2.5
METHODS=['bc_single', 'bc_local', 'bc_mean', 'bc_gaussian', 'bc_direct2_fl', 'bc_direct2_own', 'bc_direct2_shuffle', 'bc_direct2_marginal', 'bc_direct4_fl', 'bc_direct4_own', 'bc_direct4_shuffle', 'bc_direct4_marginal', 'bc_teacher2_fl', 'bc_teacher2_own', 'bc_teacher2_shuffle', 'bc_teacher2_marginal', 'bc_teacher4_fl', 'bc_teacher4_own', 'bc_teacher4_shuffle', 'bc_teacher4_marginal']
WORST_SECONDS=112.47096109390259

pending=list(METHODS);active={};summary={};seen={}
while active or pending:
    for gpu in (0,1):
        if gpu in active or not pending:continue
        remaining=SESSION_HOURS-(time.monotonic()-SESSION_T0)/3600
        if remaining<.20+1.15*WORST_SECONDS/3600:continue
        arm=pending.pop(0);folder=OUT/f'{arm}_s{SEED}';folder.mkdir(exist_ok=True)
        previous=EXPECTED.get(folder.name,{}).get('last_round',0)
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),PYTHONPATH=str(WORK),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        command=[sys.executable,str(WORK/'boundary_logging.py'),str(WORK/'papers/boundary-cells/proj/boundary_driver.py'),
            '--arm',arm,'--seed',str(SEED),'--out',str(OUT),'--rounds','20','--eval_every','1',
            '--aux_weight','.5','--temperature','1','--wandb_online','--max_hours',str(min(1.,remaining-.20))]
        if previous:command.append('--resume')
        log=open(folder/'log.txt','w');proc=subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
        active[gpu]=(arm,previous,proc,log)
        print('start utility',arm,'seed',SEED,'gpu',gpu,'parent_round',previous,flush=True)
    for gpu,(arm,previous,proc,log) in list(active.items()):
        folder=OUT/f'{arm}_s{SEED}'
        lines=(folder/'log.txt').read_text().splitlines()
        for line in lines[seen.get(arm,0):]:
            if line.startswith(('setup','r','done','WANDB','Traceback','budget')) or 'Error' in line:
                print(folder.name,'|',line,flush=True)
        seen[arm]=len(lines)
        if proc.poll() is None and time.monotonic()-SESSION_T0>=SESSION_HOURS*3600:proc.terminate()
        if proc.poll() is None and time.monotonic()-SESSION_T0>=(SESSION_HOURS+.05)*3600:proc.kill()
        if proc.poll() is None:continue
        log.close();del active[gpu]
        path=folder/'result.json';result=json.loads(path.read_text()) if path.exists() else {}
        rows=result.get('history',[])
        assert len(rows)>=previous and [r['round'] for r in rows]==list(range(1,len(rows)+1))
        summary[folder.name]=dict(rc=proc.returncode,previous_round=previous,rounds=len(rows),
            new_rounds=len(rows)-previous,complete=len(rows)==20 and proc.returncode==0,target_rounds=20)
    if not active and pending and SESSION_HOURS-(time.monotonic()-SESSION_T0)/3600<.20+1.15*WORST_SECONDS/3600:break
    if active or pending:time.sleep(2)
for arm in pending:
    name=f'{arm}_s{SEED}';previous=EXPECTED.get(name,{}).get('last_round',0)
    summary[name]=dict(previous_round=previous,rounds=previous,new_rounds=0,complete=False,
        target_rounds=20,reason='not started: session budget; preserve imported calibration if present')
(OUT/'utility_summary.json').write_text(json.dumps(summary,indent=2))
print('UTILITY BATCH',summary,flush=True)
assert len(summary)==20 and all(v.get('rc',0)==0 for v in summary.values()),'worker failure; preserve output'
print('ALL20 COMPLETE' if all(v['complete'] for v in summary.values()) else 'STAGED: utility gate remains pending',flush=True)


start utility bc_single seed 43 gpu 0 parent_round 2
start utility bc_local seed 43 gpu 1 parent_round 0
bc_single_s43 | WANDB https://wandb.ai/21522798-uit/mobihet-boundary-cells/runs/bc_single_s43-a31f484f
bc_local_s43 | WANDB https://wandb.ai/21522798-uit/mobihet-boundary-cells/runs/bc_local_s43-cb0ba7db
bc_single_s43 | setup bc_single role private rows 1000369 sec 2.4
bc_local_s43 | setup bc_local role private rows 1000369 sec 1.9
bc_single_s43 | r3 bc_single s43 private group gp 0.4604784881611292 loss 0.3321 63s
bc_local_s43 | r1 bc_local s43 private group gp 0.3804048985327022 loss 0.7992 101s
bc_single_s43 | r4 bc_single s43 private group gp 0.4613514225572196 loss 0.2685 61s
bc_single_s43 | r5 bc_single s43 private group gp 0.456645569111428 loss 0.2232 62s
bc_local_s43 | r2 bc_local s43 private group gp 0.45205381418822316 loss 0.4337 98s
bc_single_s43 | r6 bc_single s43 private group gp 0.4620192803797855 loss 0.1844 62s
bc_local_s43 | r3 bc_local s43 private group gp 0.4604